In [ ]:
import json
import numpy as np
from pathlib import Path
import os

def read_jsonl_scores(jsonl_path):
    scores_data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line.strip())
            scores_data.append({
                'sample_id': data.get('sample_id'),
                'prompt': data.get('prompt'),
                'scores': data.get('scores', {})
            })
    
    return scores_data

def calculate_win_rate(method_scores, baseline_scores, metric='avg'):
    method_dict = {item['sample_id']: item['scores'] for item in method_scores}
    baseline_dict = {item['sample_id']: item['scores'] for item in baseline_scores}
    
    common_ids = set(method_dict.keys()) & set(baseline_dict.keys())
    
    if not common_ids:
        raise ValueError("No common data ids.")
    
    wins = 0
    losses = 0
    ties = 0
    
    win_details = []
    
    for sample_id in sorted(common_ids):
        method_score = method_dict[sample_id].get(metric)
        baseline_score = baseline_dict[sample_id].get(metric)
        
        if method_score is None or baseline_score is None:
            continue
        
        if method_score > baseline_score:
            wins += 1
            result = 'win'
        elif method_score < baseline_score:
            losses += 1
            result = 'loss'
        else:
            ties += 1
            result = 'tie'
        
        win_details.append({
            'sample_id': sample_id,
            'method_score': method_score,
            'baseline_score': baseline_score,
            'diff': method_score - baseline_score,
            'result': result
        })
    
    total = wins + losses + ties
    win_rate = wins / total if total > 0 else 0
    
    score_diffs = [detail['diff'] for detail in win_details]
    
    results = {
        'win_rate': win_rate,
        'wins': wins,
        'losses': losses,
        'ties': ties,
        'total': total,
        'avg_score_diff': np.mean(score_diffs) if score_diffs else 0,
        'median_score_diff': np.median(score_diffs) if score_diffs else 0,
        'details': win_details
    }
    
    return results

def calculate_all_metrics_win_rate(method_scores, baseline_scores):
    metrics = set()
    for item in method_scores:
        metrics.update(item['scores'].keys())
    
    all_results = {}
    
    for metric in metrics:
        print(f"\nCompare metric: {metric}")
        try:
            results = calculate_win_rate(method_scores, baseline_scores, metric)
            all_results[metric] = results
            
            print(f"  Win Rate: {results['win_rate']*100:.2%}")
            print(f"  Wins: {results['wins']}, Losses: {results['losses']}, Ties: {results['ties']}")
            print(f"  Average Score Diff: {results['avg_score_diff']:.4f}")
        except Exception as e:
            print(f"  Error: {e}")
    
    return all_results

cfg_guidance=1.0
dataset="pick_a_pic_spo-backup"
base_image_dir=f"/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_{cfg_guidance}/{dataset}"

baseline_method, ckpt = "SD3.5M-DiffusionNFT-MultiReward", 0
baseline_jsonl_path = os.path.join(base_image_dir, baseline_method, f"ckpt-{ckpt}", "evaluation_results.jsonl")
print(f"{baseline_method} : {baseline_jsonl_path}")

method, ckpt = "sd3.5m-diffusionnft-multireward-next-code_reverse_score-pickscore-lr_0.0001-resize_256_crop_224-decay_type_7_0.01", 5
method_jsonl_path = os.path.join(base_image_dir, method, f"ckpt-{ckpt}", "evaluation_results.jsonl")
print(f"{method} : {method_jsonl_path}")

baseline_scores = read_jsonl_scores(baseline_jsonl_path)
method_scores = read_jsonl_scores(method_jsonl_path)
    
all_results = calculate_all_metrics_win_rate(method_scores, baseline_scores)

SD3.5M-DiffusionNFT-MultiReward : /data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pick_a_pic_spo-backup/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/evaluation_results.jsonl
sd3.5m-diffusionnft-multireward-next-code_reverse_score-pickscore-lr_0.0001-resize_256_crop_224-decay_type_7_0.01 : /data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pick_a_pic_spo-backup/sd3.5m-diffusionnft-multireward-next-code_reverse_score-pickscore-lr_0.0001-resize_256_crop_224-decay_type_7_0.01/ckpt-5/evaluation_results.jsonl


FileNotFoundError: [Errno 2] No such file or directory: '/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pick_a_pic_spo-backup/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/evaluation_results.jsonl'